# 真实论文抽取数据准备

从 `data/raw/e08-paper-knowledge/papers.yml` 加载六篇候选，批量固定材料。
本 notebook **只准备材料**；后续逐篇阅读、实体消歧和图谱写入另行开展。

1. 验证清单里的 arXiv / S2 身份，保存 S2 元数据。
2. 分页获取 **references（当前论文引用的文献）**，保留原始边字段。
3. 下载固定版本的 arXiv 官方 HTML，用本仓库 `arxiv2md` 转换，下载并本地化图片。
4. 验收哈希、分页终点、Markdown 与图片完整性，写出 `status.yml`；不完整则最后一格报错。

结构化材料统一使用 `.yml`；原始 HTML、Markdown 和图片保留各自原生格式：

```text
data/raw/e08-paper-knowledge/
  papers.yml                     # 清单：身份与固定 arXiv 版本
  status.yml                     # 当前材料的验收结果与失败清单
  runs/<run_id>.yml               # 每次执行配置、每次请求/重试（无密钥）
  papers/<key>/
    s2/paper.yml                 # 原始 API payload + 请求溯源
    s2/references.yml            # 从分页材料聚合的参考文献
    s2/references_pages/*.yml    # 分页断点，保留每页原始响应
    source.html
    source.yml                  # HTML 来源与哈希
    paper.md
    document.yml                # 转换器版本、章节、HTML/MD 哈希与验收计数
    images.yml                  # 图片 URL、本地路径、哈希与失败情况
    images/*
```

重跑复用已成功并通过校验的下载，失败只补缺失部分；清单不变时不刷新 S2 快照。
本流程不获取施引文献列表，也不将它作为验收条件。早先运行留下的施引文件不参与后续流程。
标准 S2 元数据中的 citationCount 仅作为原始响应的一部分保存，不据此展开邻居。
参考文献清单只是候选材料；后续按阅读需要决定哪些 Paper 建立占位、哪些进一步抽取，不全部自动入图。
分页依照 API 的 `next` 直到终点，不使用单篇端点的嵌套 1,000 条截断列表。
S2 的计数与分页数据可能因索引更新而不同，记录差异，不把差异抹平。
分页材料保存起止获取时间，**不宣称整个数据集是服务端原子快照**。
S2 的引用和元数据也不等于已选 arXiv 版本的逐条事实，二者分别保存。

需要刷新快照或改论文版本时，请使用新的材料目录，避免混合旧页和新页。
从仓库共享 `.venv` 的新内核自上而下运行；所有步骤都可以补跑。
API 依据：[Semantic Scholar Academic Graph API](https://api.semanticscholar.org/api-docs/graph)。

## 加载真实论文清单与运行参数

In [ ]:
import asyncio
import hashlib
import json
import logging
import os
import re
import subprocess
import time
import unicodedata
import xml.etree.ElementTree as ET
from datetime import datetime, timezone
from email.utils import parsedate_to_datetime
from pathlib import Path
from urllib.parse import urlparse

import httpx
import pandas as pd
import yaml
from bs4 import BeautifulSoup

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'AGENTS.md').is_file())
ROOT = REPO / 'data/raw/e08-paper-knowledge'
MANIFEST_PATH = ROOT / 'papers.yml'
S2_BASE = 'https://api.semanticscholar.org/graph/v1'
PAGE_SIZE = 1000
MAX_ATTEMPTS = 6
TIMEOUT = httpx.Timeout(90.0, connect=20.0)
MIN_INTERVAL = {'api.semanticscholar.org': 1.3, 'arxiv.org': 1.0}
RETRY_STATUS = {408, 425, 429, 500, 502, 503, 504}
USER_AGENT = 'CachePlan-E08-material-preparation/1.0'

def utcnow():
    return datetime.now(timezone.utc).isoformat(timespec='seconds')

def sha256(data):
    return hashlib.sha256(data).hexdigest()

def payload_hash(value):
    return sha256(json.dumps(value, sort_keys=True, ensure_ascii=False).encode('utf-8'))

def read_yml(path):
    return yaml.safe_load(path.read_text(encoding='utf-8'))

def atomic_bytes(path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.part')
    tmp.write_bytes(data)
    tmp.replace(path)

def write_yml(path, value):
    atomic_bytes(path, yaml.safe_dump(value, allow_unicode=True, sort_keys=False,
                                    width=110).encode('utf-8'))

def relative(path):
    return path.relative_to(ROOT).as_posix()

def normalize_title(value):
    value = unicodedata.normalize('NFKC', value).casefold()
    return ''.join(c for c in value if c.isalnum())

def api_key():
    if os.environ.get('S2K'):
        return os.environ['S2K'], 'environment:S2K'
    for path in [REPO / 'experiments/e08/.env', REPO / '.env']:
        if path.is_file():
            for raw in path.read_text().splitlines():
                k, sep, v = raw.strip().removeprefix('export ').partition('=')
                if sep and k.strip() == 'S2K' and v.strip():
                    return v.strip().strip('"\''), str(path.relative_to(REPO))
    return None, 'anonymous'

S2_KEY, KEY_ORIGIN = api_key()
manifest = read_yml(MANIFEST_PATH)
PAPERS = manifest['papers']
assert PAPERS and len({p['key'] for p in PAPERS}) == len(PAPERS)
assert len({p['arxiv_id'] for p in PAPERS}) == len(PAPERS)
for p in PAPERS:
    assert re.fullmatch(r'[A-Za-z0-9_-]+', p['key'])
    assert re.fullmatch(r'\d{4}\.\d{4,5}', p['arxiv_id'])
    assert re.fullmatch(r'v[1-9]\d*', p['arxiv_version'])
    assert not p.get('s2_id') or re.fullmatch(r'[0-9a-f]{40}', p['s2_id'])

RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
RUN_PATH = ROOT / 'runs' / f'{RUN_ID}.yml'
ERRORS = []
RUN = {'run_id': RUN_ID, 'started_at': utcnow(),
       'manifest_sha256': sha256(MANIFEST_PATH.read_bytes()),
       'config': {'page_size': PAGE_SIZE, 'max_attempts': MAX_ATTEMPTS,
                  'citation_scope': 'references_only',
                  'timeout_read_seconds': TIMEOUT.read, 'min_interval_seconds': MIN_INTERVAL,
                  'key_origin': KEY_ORIGIN, 's2_key_present': bool(S2_KEY)},
       'requests': []}
write_yml(RUN_PATH, RUN)
print(f'材料目录: {ROOT.relative_to(REPO)} | {len(PAPERS)} 篇')
print(f'S2 key: {"已配置" if S2_KEY else "未配置，匿名限额可能较紧"}；来源: {KEY_ORIGIN}')
display(pd.DataFrame(PAPERS))

## 网络请求：限速、重试与溯源

所有 S2、HTML、图片请求经过同一个入口。对超时、连接失败、429、部分 5xx，以及
HTTP 200 但内容无效的响应重试；遵守 `Retry-After`，否则确定性指数退避。
其他 4xx 直接记录失败。HTTP 200 的验证码页不能当作论文或图片保存。
每次尝试立即写入运行记录；不记录请求头、密钥、代理地址或响应正文。

In [ ]:
_last_request = {}

def clean_error(exc):
    message = f'{type(exc).__name__}: {exc}'
    return message.replace(S2_KEY, '[REDACTED]') if S2_KEY else message

def record_error(paper, stage, exc):
    item = {'paper': paper['key'], 'stage': stage, 'error': clean_error(exc), 'at': utcnow()}
    ERRORS.append(item)
    print(f"FAIL {paper['key']} / {stage}: {item['error']}", flush=True)

def retry_delay(response, attempt):
    delay = min(60.0, 2.0 ** attempt)
    raw = response.headers.get('retry-after') if response is not None else None
    if raw:
        try:
            seconds = float(raw)
        except ValueError:
            try:
                seconds = (parsedate_to_datetime(raw) - datetime.now(timezone.utc)).total_seconds()
            except (TypeError, ValueError, OverflowError):
                seconds = 0
        delay = max(delay, seconds)
    return delay

async def request(client, url, *, params=None, validate=None):
    host = urlparse(url).hostname
    headers = {'User-Agent': USER_AGENT}
    # 密钥仅发给 S2 API；S2 不自动跨域重定向。
    if host == 'api.semanticscholar.org' and S2_KEY:
        headers['x-api-key'] = S2_KEY
    for attempt in range(1, MAX_ATTEMPTS + 1):
        gap = MIN_INTERVAL.get(host, 1.0) - (time.monotonic() - _last_request.get(host, 0))
        if gap > 0:
            await asyncio.sleep(gap)
        _last_request[host] = time.monotonic()
        start = time.monotonic()
        response = None
        event = {'at': utcnow(), 'url': url, 'params': params or {}, 'attempt': attempt}
        try:
            response = await client.get(url, params=params, headers=headers,
                                        follow_redirects=host != 'api.semanticscholar.org')
            response.raise_for_status()
            value = validate(response) if validate else response.content
        except (httpx.HTTPError, ValueError) as exc:
            retryable = (response is None or response.status_code in RETRY_STATUS
                         or 200 <= response.status_code < 300)
            retry = retryable and attempt < MAX_ATTEMPTS
            delay = retry_delay(response, attempt) if retry else None
            event.update(status=response.status_code if response is not None else None,
                         outcome='retry' if retry else 'failed', error=clean_error(exc),
                         retry_after_seconds=delay,
                         elapsed_seconds=round(time.monotonic() - start, 3))
            RUN['requests'].append(event)
            write_yml(RUN_PATH, RUN)
            if not retry:
                raise RuntimeError(f'Request failed: {url}: {clean_error(exc)}') from None
            print(f"retry {attempt}/{MAX_ATTEMPTS} HTTP {event['status']} {url} ({delay:.1f}s)", flush=True)
            await asyncio.sleep(delay)
        else:
            event.update(status=response.status_code, outcome='ok', final_url=str(response.url),
                         bytes=len(response.content),
                         elapsed_seconds=round(time.monotonic() - start, 3))
            RUN['requests'].append(event)
            write_yml(RUN_PATH, RUN)
            return value, event
    raise RuntimeError('Unreachable')

def json_object(response):
    result = response.json()
    if not isinstance(result, dict) or 'error' in result:
        raise ValueError('S2 returned an invalid JSON object')
    return result

async def snapshot(client, path, url, params, validate_payload):
    if path.is_file():
        saved = read_yml(path)
        if saved['_request']['url'] != url or saved['_request']['params'] != params:
            raise ValueError(f'缓存请求参数变化，请使用新的材料目录: {relative(path)}')
        if payload_hash(saved['data']) != saved['_request']['payload_sha256']:
            raise ValueError(f'缓存校验失败: {relative(path)}')
        validate_payload(saved['data'])
        return saved
    def validate(response):
        value = json_object(response)
        validate_payload(value)
        return value
    data, event = await request(client, url, params=params, validate=validate)
    saved = {'_request': {'url': url, 'params': params, 'fetched_at': event['at'],
                         'final_url': event['final_url'], 'payload_sha256': payload_hash(data)},
             'data': data}
    write_yml(path, saved)
    return saved

## 从 S2 获取元数据和分页参考文献

清单中的 S2 ID 来自预先核验；首次仍按 arXiv ID 请求并核对返回的 S2 ID、arXiv ID 和标题。
未填写 S2 ID 时允许解析后回填。参考文献逐页断点续传；一篇失败不阻止其他论文及 HTML 准备。
邻居保留身份、题名、作者、日期和入口，不递归下载它们的全文。
`contexts` / `intents` / `contextsWithIntent` 是 S2 给出的记录，不替代正文证据。

In [ ]:
SELF_FIELDS = ','.join([
    'paperId', 'corpusId', 'externalIds', 'url', 'title', 'abstract', 'authors',
    'venue', 'publicationVenue', 'year', 'publicationDate', 'journal', 'publicationTypes',
    'fieldsOfStudy', 's2FieldsOfStudy', 'referenceCount', 'citationCount',
    'influentialCitationCount', 'isOpenAccess', 'openAccessPdf', 'citationStyles', 'tldr',
])
EDGE_FIELDS = ','.join([
    'contexts', 'intents', 'contextsWithIntent', 'isInfluential',
    'paperId', 'corpusId', 'externalIds', 'title', 'authors', 'year', 'publicationDate',
    'venue', 'url', 'isOpenAccess', 'openAccessPdf',
])

def validate_paper(data, paper):
    if not re.fullmatch(r'[0-9a-f]{40}', data.get('paperId', '')):
        raise ValueError('Missing S2 paperId')
    if paper.get('s2_id') and data['paperId'] != paper['s2_id']:
        raise ValueError('S2 ID disagrees with the manifest')
    if (data.get('externalIds') or {}).get('ArXiv') != paper['arxiv_id']:
        raise ValueError('S2 arXiv ID disagrees with the manifest')
    if normalize_title(data.get('title', '')) != normalize_title(paper['title']):
        raise ValueError('S2 title disagrees with the manifest')

def validate_page(data, offset):
    if data.get('offset') != offset or not isinstance(data.get('data'), list):
        raise ValueError(f'Invalid references page at offset {offset}')
    nxt = data.get('next')
    if nxt is not None and (type(nxt) is not int or nxt <= offset or not data['data']):
        raise ValueError(f'Invalid next offset: {nxt}')
    field = 'citedPaper'
    for edge in data['data']:
        if not isinstance(edge, dict) or field not in edge:
            raise ValueError(f'Missing {field} in an edge')

async def get_references(client, paper, metadata):
    direction = 'references'
    out = ROOT / 'papers' / paper['key'] / 's2'
    offset, pages, entries = 0, [], []
    while True:
        url = f"{S2_BASE}/paper/{paper['s2_id']}/{direction}"
        params = {'fields': EDGE_FIELDS, 'offset': offset, 'limit': PAGE_SIZE}
        path = out / f'{direction}_pages' / f'{offset:06d}.yml'
        page = await snapshot(client, path, url, params,
                              lambda d: validate_page(d, offset))
        pages.append({'path': relative(path), 'offset': offset,
                      'next': page['data'].get('next'),
                      'fetched_at': page['_request']['fetched_at'],
                      'sha256': sha256(path.read_bytes())})
        entries.extend(page['data']['data'])
        print(f"  {paper['key']} {direction}: {len(entries)} 条", flush=True)
        offset = page['data'].get('next')
        if offset is None:
            break
    endpoint = 'citedPaper'
    ids = [(e.get(endpoint) or {}).get('paperId') for e in entries]
    known = [i for i in ids if i]
    expected = metadata.get('referenceCount')
    aggregate = {
        '_fetch': {'s2_id': paper['s2_id'], 'direction': direction,
                   'complete': True, 'completion_basis': 'API next exhausted',
                   'first_fetched_at': min(p['fetched_at'] for p in pages),
                   'last_fetched_at': max(p['fetched_at'] for p in pages),
                   'pages': pages, 'rows': len(entries), 'unique_known_papers': len(set(known)),
                   'null_paper_ids': len(ids) - len(known),
                   'duplicate_known_ids': len(known) - len(set(known)),
                   'metadata_reported_count': expected,
                   'count_matches_metadata': len(entries) == expected,
                   'payload_sha256': payload_hash(entries),
                   'note': 'S2 分页响应快照；next 耗尽不证明引文语义完整或对应指定 arXiv 版本。'},
        'data': entries,
    }
    write_yml(out / f'{direction}.yml', aggregate)

async with httpx.AsyncClient(timeout=TIMEOUT) as client:
    for paper in PAPERS:
        out = ROOT / 'papers' / paper['key'] / 's2'
        try:
            saved = await snapshot(client, out / 'paper.yml',
                f"{S2_BASE}/paper/ARXIV:{paper['arxiv_id']}", {'fields': SELF_FIELDS},
                lambda d: validate_paper(d, paper))
            if not paper.get('s2_id'):
                paper['s2_id'] = saved['data']['paperId']
                write_yml(MANIFEST_PATH, manifest)
            print(f"S2 {paper['key']} {paper['s2_id']}", flush=True)
        except Exception as exc:
            record_error(paper, 's2_metadata', exc)
            continue
        try:
            await get_references(client, paper, saved['data'])
        except Exception as exc:
            record_error(paper, 's2_references', exc)

## 使用 arxiv2md 准备 HTML、Markdown 和本地图片

复用 `03_arxiv2md.ipynb` 的转换路径：解析章节 → 转换各节 → 拼接全文。
保留参考文献、行内引用、公式、表格及图片；本次只使用清单中的官方 HTML，不偷偷换版本。
抓取层统一使用上面的重试入口，转换调用 `packages/arxiv2md` 的本地函数。
关闭可选的 token 估计，避免 tokenizer 隐式下载词表。

图片逐张保存、核验哈希；中断后无需重下已完成图片。失败图片保留远程链接并使最终验收失败。
固定源材料与转换器源码哈希。相同输入重跑时校验已有 Markdown，避免后续原文 anchor 悄悄漂移。

In [ ]:
from arxiv2md.html_parser import parse_arxiv_html
from arxiv2md.markdown import convert_fragment_to_markdown, unlink_dangling_fragment_links
from arxiv2md.images import _MD_IMAGE_RE, _image_filename
import arxiv2md.output_formatter as formatter

logging.getLogger('httpx').setLevel(logging.WARNING)
formatter.tiktoken = None  # 只关闭可选估计，不修改转换内容。
CONVERTER_DIR = REPO / 'packages/arxiv2md/src/arxiv2md'
CONVERTER_SHA = payload_hash({str(p.relative_to(CONVERTER_DIR)): sha256(p.read_bytes())
                              for p in sorted(CONVERTER_DIR.rglob('*.py'))})
GIT_SHA = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
CONVERT_OPTIONS = {'include_toc': False, 'include_frontmatter': True,
                   'remove_references': False, 'remove_inline_citations': False}

def validate_html(raw, paper):
    html = raw.decode('utf-8')
    soup = BeautifulSoup(html, 'html.parser')
    article = soup.select_one('article.ltx_document')
    title = soup.select_one('h1.ltx_title_document')
    if article is None or title is None or len(article.get_text()) < 1000:
        raise ValueError('Not a full arXiv LaTeXML article (possibly an error/challenge page)')
    if normalize_title(title.get_text(' ', strip=True)) != normalize_title(paper['title']):
        raise ValueError('HTML title disagrees with the manifest')
    return html

async def get_html(client, paper, out):
    version_id = paper['arxiv_id'] + paper['arxiv_version']
    url = f'https://arxiv.org/html/{version_id}'
    path, meta_path = out / 'source.html', out / 'source.yml'
    if path.is_file() and meta_path.is_file():
        meta = read_yml(meta_path)
        if meta['requested_url'] != url or sha256(path.read_bytes()) != meta['sha256']:
            raise ValueError('HTML 缓存版本或哈希不符，请使用新目录或修复损坏文件')
        return validate_html(path.read_bytes(), paper), meta
    def validate(response):
        if urlparse(str(response.url)).path.rstrip('/') != f'/html/{version_id}':
            raise ValueError('HTML request redirected to a different paper/version')
        validate_html(response.content, paper)
        return response.content
    raw, event = await request(client, url, validate=validate)
    meta = {'arxiv_id': paper['arxiv_id'], 'version': paper['arxiv_version'],
            'requested_url': url, 'source_url': event['final_url'],
            'fetched_at': event['at'], 'sha256': sha256(raw), 'bytes': len(raw)}
    atomic_bytes(path, raw)
    write_yml(meta_path, meta)
    return raw.decode('utf-8'), meta

def convert_html(html, paper, source_url):
    parsed = parse_arxiv_html(html)
    if not parsed.sections or not parsed.abstract:
        raise ValueError('Missing article sections or abstract')
    outline = []
    def populate(section):
        outline.append({'title': section.title, 'anchor': section.anchor, 'level': section.level})
        if section.html:
            section.markdown = convert_fragment_to_markdown(section.html, base_url=source_url)
        for child in section.children:
            populate(child)
    for section in parsed.sections:
        populate(section)
    result = formatter.format_paper(arxiv_id=paper['arxiv_id'], version=paper['arxiv_version'],
        title=parsed.title, authors=parsed.authors, abstract=parsed.abstract, sections=parsed.sections,
        include_toc=False, include_abstract_in_tree=True, include_frontmatter=True)
    # frontmatter 自身没有结尾换行，必须显式与正文分隔。
    content = result.frontmatter + '\n\n' + unlink_dangling_fragment_links(result.content) + '\n'
    if len(content) < 2000:
        raise ValueError('Converted Markdown is unexpectedly short')
    return content, parsed, outline

def validate_image(response):
    raw = response.content
    if not raw:
        raise ValueError('Empty image')
    is_svg = False
    if b'<svg' in raw[:2000]:
        try:
            is_svg = ET.fromstring(raw).tag.rsplit('}', 1)[-1] == 'svg'
        except ET.ParseError as exc:
            raise ValueError('Malformed SVG') from exc
    raster = (raw.startswith(b'\x89PNG\r\n\x1a\n') and raw.endswith(b'IEND\xaeB`\x82')
              or raw.startswith(b'\xff\xd8') and raw.rstrip().endswith(b'\xff\xd9')
              or raw.startswith((b'GIF87a', b'GIF89a'))
              or raw.startswith(b'RIFF') and raw[8:12] == b'WEBP')
    if not (is_svg or raster):
        raise ValueError('Response is not a supported complete image (possibly HTML)')
    return raw

async def get_images(client, paper, out, content):
    urls = list(dict.fromkeys(m.group(2) for m in _MD_IMAGE_RE.finditer(content)))
    used_names = set()
    names = {url: _image_filename(url, used_names) for url in urls}
    path = out / 'images.yml'
    saved = read_yml(path) if path.is_file() else {'items': {}}
    local = {}
    for url in urls:
        dest = out / 'images' / names[url]
        previous = saved['items'].get(url, {})
        if (previous.get('status') == 'ok' and dest.is_file()
                and sha256(dest.read_bytes()) == previous.get('sha256')):
            local[url] = f'images/{names[url]}'
            continue
        try:
            raw, event = await request(client, url, validate=validate_image)
            atomic_bytes(dest, raw)
            local[url] = f'images/{names[url]}'
            saved['items'][url] = {'status': 'ok', 'path': local[url], 'bytes': len(raw),
                                   'sha256': sha256(raw), 'fetched_at': event['at']}
        except Exception as exc:
            saved['items'][url] = {'status': 'failed', 'error': clean_error(exc)}
            record_error(paper, 'image', exc)
        write_yml(path, saved)
    saved['expected_urls'] = urls
    saved['complete'] = len(local) == len(urls)
    write_yml(path, saved)
    localized = _MD_IMAGE_RE.sub(lambda m: f'![{m.group(1)}]({local.get(m.group(2), m.group(2))})', content)
    return localized, len(urls), len(local)

async with httpx.AsyncClient(timeout=TIMEOUT) as client:
    for paper in PAPERS:
        out = ROOT / 'papers' / paper['key']
        try:
            html, source = await get_html(client, paper, out)
            previous = read_yml(out / 'document.yml') if (out / 'document.yml').is_file() else None
            if previous and previous['complete']:
                if (previous['converter']['source_sha256'] != CONVERTER_SHA
                        or previous['html_sha256'] != source['sha256']):
                    raise ValueError('源 HTML 或转换器变化，请使用新目录，避免修改已固定的原文')
                if sha256((out / 'paper.md').read_bytes()) != previous['markdown_sha256']:
                    raise ValueError('已固定 Markdown 的哈希不符')
            content, parsed, outline = convert_html(html, paper, source['source_url'])
            bibliography_ids = [item['id'] for item in BeautifulSoup(html, 'html.parser').select('.ltx_bibitem[id]')]
            markdown_anchors = set(re.findall(r'<a id="([^"]+)"', content))
            if not all(anchor in markdown_anchors for anchor in bibliography_ids):
                raise ValueError('部分 HTML 参考文献锚点在 Markdown 中丢失')
            content, total, done = await get_images(client, paper, out, content)
            encoded = content.encode('utf-8')
            if previous and previous['complete'] and sha256(encoded) != previous['markdown_sha256']:
                raise ValueError('相同输入的 Markdown 重建结果不一致，未覆盖原文件')
            atomic_bytes(out / 'paper.md', encoded)
            meta = {'arxiv_id': paper['arxiv_id'], 'version': paper['arxiv_version'],
                    'title': parsed.title, 'authors_from_html': parsed.authors,
                    'source_url': source['source_url'], 'html_sha256': source['sha256'],
                    'markdown_path': relative(out / 'paper.md'), 'markdown_sha256': sha256(encoded),
                    'converter': {'name': 'arxiv2md (vendored)', 'git_sha': GIT_SHA,
                                  'source_sha256': CONVERTER_SHA, 'options': CONVERT_OPTIONS},
                    'outline': outline, 'complete': total == done,
                    'counts': {'markdown_chars': len(content), 'sections': len(outline),
                               'html_references': len(bibliography_ids),
                               'table_rows': sum(line.startswith('|') for line in content.splitlines()),
                               'images_total': total, 'images_local': done,
                               'remote_images': len(_MD_IMAGE_RE.findall(content))}}
            write_yml(out / 'document.yml', meta)
            print(f"HTML/MD {paper['key']}: {len(content):,} chars, {len(outline)} sections, images {done}/{total}", flush=True)
        except Exception as exc:
            record_error(paper, 'arxiv2md', exc)

## 验收与入口索引

验收读取已落盘材料，不仅看进程是否成功。检查分页链直到终点、聚合内容与原始页一致、
身份一致、HTML/MD/图片哈希，以及图片全部本地化。`api_complete` 表示已取完 API 可返回的页；
语义正确性、S2 索引覆盖率及图表转换质量仍需要后续阅读检查。

`status.yml` 是后续抽取的材料入口；`ready: true` 才表示这一篇本轮准备完成。
若失败，先看 `errors` 与 `runs/<run_id>.yml`，修复后从头重跑即可补齐。

In [ ]:
def verify_references(out, paper):
    direction = 'references'
    saved = read_yml(out / 's2' / f'{direction}.yml')
    meta = saved['_fetch']
    assert meta['complete'] and meta['s2_id'] == paper['s2_id']
    entries, offset = [], 0
    for page in meta['pages']:
        path = ROOT / page['path']
        assert sha256(path.read_bytes()) == page['sha256']
        value = read_yml(path)
        assert value['_request']['url'] == f"{S2_BASE}/paper/{paper['s2_id']}/{direction}"
        assert value['_request']['params'] == {'fields': EDGE_FIELDS, 'offset': offset, 'limit': PAGE_SIZE}
        assert payload_hash(value['data']) == value['_request']['payload_sha256']
        validate_page(value['data'], offset)
        entries.extend(value['data']['data'])
        offset = value['data'].get('next')
    assert meta['pages'] and offset is None
    assert entries == saved['data'] and len(entries) == meta['rows']
    assert payload_hash(entries) == meta['payload_sha256']
    return meta

rows = []
for paper in PAPERS:
    out = ROOT / 'papers' / paper['key']
    row = {'key': paper['key'], 'arxiv': paper['arxiv_id'] + paper['arxiv_version'],
           's2_id': paper.get('s2_id'), 'ready': False, 's2_ready': False, 'document_ready': False}
    try:
        metadata = read_yml(out / 's2/paper.yml')
        validate_paper(metadata['data'], paper)
        assert payload_hash(metadata['data']) == metadata['_request']['payload_sha256']
        refs = verify_references(out, paper)
        row.update(s2_ready=True, references=refs['rows'],
                   reference_count_matches=refs['count_matches_metadata'],
                   metadata_path=relative(out / 's2/paper.yml'),
                   references_path=relative(out / 's2/references.yml'))
    except Exception as exc:
        record_error(paper, 'verify_s2', exc)
    try:
        doc, source, images = [read_yml(out / name) for name in ['document.yml', 'source.yml', 'images.yml']]
        assert doc['complete'] and images['complete']
        assert doc['arxiv_id'] == paper['arxiv_id'] and doc['version'] == paper['arxiv_version']
        assert source['requested_url'] == f"https://arxiv.org/html/{row['arxiv']}"
        assert sha256((out / 'source.html').read_bytes()) == source['sha256'] == doc['html_sha256']
        assert sha256((out / 'paper.md').read_bytes()) == doc['markdown_sha256']
        assert doc['converter']['source_sha256'] == CONVERTER_SHA
        markdown = (out / 'paper.md').read_text(encoding='utf-8')
        assert not _MD_IMAGE_RE.search(markdown)
        for url in images['expected_urls']:
            item = images['items'][url]
            assert item['status'] == 'ok'
            assert sha256((out / item['path']).read_bytes()) == item['sha256']
            assert f"]({item['path']})" in markdown
        assert doc['counts']['images_total'] == len(images['expected_urls'])
        row.update(document_ready=True, markdown_chars=doc['counts']['markdown_chars'],
                   html_references=doc['counts']['html_references'],
                   images=doc['counts']['images_local'], markdown_path=doc['markdown_path'],
                   document_path=relative(out / 'document.yml'))
    except Exception as exc:
        record_error(paper, 'verify_document', exc)
    row['ready'] = row['s2_ready'] and row['document_ready'] and not any(e['paper'] == paper['key'] for e in ERRORS)
    rows.append(row)

REPORT = {'run_id': RUN_ID, 'checked_at': utcnow(),
          'citation_scope': 'references_only',
          'manifest_sha256': sha256(MANIFEST_PATH.read_bytes()),
          'all_ready': all(r['ready'] for r in rows), 'papers': rows, 'errors': ERRORS,
          'request_log': relative(RUN_PATH),
          'note': '材料准备验收，不代表已抽取、已入图或已验证论文主张。'}
write_yml(ROOT / 'status.yml', REPORT)
RUN.update(finished_at=utcnow(), all_ready=REPORT['all_ready'], errors=ERRORS,
           manifest_final_sha256=REPORT['manifest_sha256'])
write_yml(RUN_PATH, RUN)
display(pd.DataFrame(rows).reindex(columns=['key', 'ready', 'references', 'markdown_chars', 'images']))
print(f"本轮网络尝试 {len(RUN['requests'])} 次；失败记录 {len(ERRORS)} 条；入口 {ROOT / 'status.yml'}")
assert REPORT['all_ready'], '材料尚未准备齐全；检查 status.yml 和对应运行日志后重跑。'